# 6.03 流匹配：π0 的动作头

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/06_generative_policy/6.03_flow_matching.ipynb)


## 为什么要学

π0、π0.5、GR00T N1、SmolVLA 都用**流匹配**（Flow Matching）生成动作。和扩散相比，它：

- 公式更简单（一行加权平均 + 一行 MSE）；
- 路径更"直"，**10 步 Euler 积分**就能得到高质量样本（π0 默认 `num_steps=10`）；
- 与 Transformer 动作专家配合得很好。

本课**严格按照 openpi 中 `pi0_pytorch.py` 的约定**实现流匹配。学完后，你会发现 π0 的 `forward()` 和 `sample_actions()` 中与流匹配相关的代码，你都能逐行看懂。

---

## 1. 核心思想：学一个速度场

想象从噪声分布到数据分布之间有一个"流场"：每个点都有一个速度 $\mathbf v(\mathbf x,t)$。从噪声出发、沿着速度场走，就会流到数据上。这就是一个常微分方程（ODE）：

$$
\frac{d\mathbf x_t}{dt}=\mathbf v_\theta(\mathbf x_t,t)
$$

问题：怎么学到这个速度场？

### openpi 的概率路径（直线插值）

对每一对（数据 $\mathbf x$, 噪声 $\boldsymbol\epsilon$），定义一条**直线路径**（openpi 约定：$t=1$ 是噪声，$t=0$ 是数据）：

$$
\boxed{\mathbf x_t=t\,\boldsymbol\epsilon+(1-t)\,\mathbf x}
$$

沿这条直线运动的速度是常数：

$$
\mathbf u_t=\frac{d\mathbf x_t}{dt}=\boldsymbol\epsilon-\mathbf x
$$

对照 openpi 源码 `PI0Pytorch.forward`：

```py
x_t = time_expanded * noise + (1 - time_expanded) * actions
u_t = noise - actions
...
v_t = self.action_out_proj(suffix_out)
return F.mse_loss(u_t, v_t, reduction="none")
```

### 训练目标：条件流匹配

$$
\mathcal L=\mathbb E_{t,\mathbf x,\boldsymbol\epsilon}\Big[\big\|\mathbf v_\theta(\mathbf x_t,t,\mathbf o)-(\boldsymbol\epsilon-\mathbf x)\big\|^2\Big]
$$

**一个关键的理论结果**：同一个 $\mathbf x_t$ 可能来自很多不同的 $(\mathbf x,\boldsymbol\epsilon)$ 对，所以"目标" $\boldsymbol\epsilon-\mathbf x$ 是有噪声的。但 MSE 回归学到的是**条件期望**：

$$
\mathbf v^*(\mathbf x_t,t)=\mathbb E[\boldsymbol\epsilon-\mathbf x\mid\mathbf x_t]
$$

而 Lipman 等人证明了：**沿着这个"平均速度场"积分，得到的正好是数据分布**。所以只需要做一个普通的回归，就能学到一个生成模型。

### 推理：从噪声积分到数据

从 $\mathbf x_1=\boldsymbol\epsilon\sim\mathcal N(0,I)$ 出发，用 Euler 法从 $t=1$ 积分到 $t=0$（步长 $\Delta t=-1/N$）：

$$
\mathbf x_{t+\Delta t}=\mathbf x_t+\Delta t\cdot\mathbf v_\theta(\mathbf x_t,t)
$$

对照 openpi 的 `sample_actions`：

```py
dt = -1.0 / num_steps
x_t = noise
time = torch.tensor(1.0)
while time >= -dt / 2:
    v_t = self.denoise_step(state, prefix_pad_masks, past_key_values, x_t, expanded_time)
    x_t = x_t + dt * v_t
    time += dt
return x_t
```

（`while time >= -dt/2` 而不是 `time > 0`，是为了避免浮点误差导致少走或多走一步。）

---

## 2. 在二维数据上实现

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.models.flow import ChunkDenoiser
from vla_ladder.utils import set_seed, get_device

device = get_device()

def make_ring_of_gaussians(n, k=6, r=1.0, std=0.08, seed=0):
    g = np.random.default_rng(seed)
    centers = np.stack([r * np.cos(2 * np.pi * np.arange(k) / k), r * np.sin(2 * np.pi * np.arange(k) / k)], 1)
    return torch.tensor(centers[g.integers(0, k, n)] + std * g.standard_normal((n, 2)), dtype=torch.float32)

data = make_ring_of_gaussians(20000).to(device)

### 练习 1：π0 的时间采样

$$
t = 0.999\cdot\tau+0.001,\qquad\tau\sim\text{Beta}(1.5,1)
$$

In [ ]:
def sample_time(B):
    # TODO: torch.distributions.Beta(1.5, 1.0).sample((B,))，再线性变换；放到 device 上
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def sample_time(B):
    tau = torch.distributions.Beta(torch.tensor(1.5), torch.tensor(1.0)).sample((B,))
    return (tau * 0.999 + 0.001).to(device)

In [ ]:
# ✅ 自测
ts = sample_time(100000).cpu()
assert ts.min() >= 0.001 and ts.max() <= 1.0
print(f"E[t] = {ts.mean():.3f}（Beta(1.5,1) 的均值 0.6，变换后 ≈ 0.6）；t>0.5 的比例 {(ts > 0.5).float().mean():.2f}")
assert abs(ts.mean() - (0.6 * 0.999 + 0.001)) < 0.01
plt.figure(figsize=(4, 2.5)); plt.hist(ts.numpy(), bins=50, density=True); plt.title("pi0 time sampling"); plt.xlabel("t (1 = noise)"); plt.show()
print("通过 ✅")

> **为什么偏向 $t\approx1$（噪声端）？**（回答 1.05 课的思考题）在噪声多的区域，网络需要决定"往哪个模式走"，这是最难也最关键的部分；而 $t\approx0$ 附近只是在已经确定的模式内部做小修正，相对简单。π0 论文指出，与图像生成不同，**动作生成在给定观测后的分布更集中**，因此把训练算力更多地放在高噪声区域更有用。

### 练习 2：流匹配损失（openpi 约定）

In [ ]:
def fm_loss(model, x, obs):
    """x: (B, H, A) 数据（动作块）；obs: (B, obs_dim)"""
    # TODO:
    # noise = randn_like(x)；t = sample_time(B)；te = t[:, None, None]
    # x_t = te * noise + (1 - te) * x；u_t = noise - x
    # v_t = model(x_t, t, obs)；return MSE(v_t, u_t)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def fm_loss(model, x, obs):
    """x: (B, H, A) 数据（动作块）；obs: (B, obs_dim)"""
    noise = torch.randn_like(x)
    t = sample_time(x.shape[0])
    te = t[:, None, None]
    x_t = te * noise + (1 - te) * x
    u_t = noise - x
    v_t = model(x_t, t, obs)
    return ((v_t - u_t) ** 2).mean()

In [ ]:
# ✅ 训练二维流匹配模型
set_seed(0)
model = ChunkDenoiser(obs_dim=1, act_dim=2, horizon=1).to(device)
dummy = lambda B: torch.zeros(B, 1, device=device)
opt = torch.optim.AdamW(model.parameters(), 1e-3)
steps = 4000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
for step in range(steps):
    x = data[torch.randint(0, len(data), (512,), device=device)][:, None, :]
    loss = fm_loss(model, x, dummy(512))
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 1000 == 0:
        print(f"step {step}  loss {loss.item():.4f}")
model.eval()
print("通过 ✅")

> 流匹配的 loss **不会降到 0**：同一个 $\mathbf x_t$ 对应很多个目标 $\boldsymbol\epsilon-\mathbf x$，回归只能学到它们的平均。看 loss 判断不了生成质量，要看采样结果。

### 练习 3：Euler 采样（与 openpi 的 `sample_actions` 写法一致）

In [ ]:
@torch.no_grad()
def euler_sample(model, obs, shape, num_steps=10, return_traj=False):
    dt = -1.0 / num_steps
    x_t = torch.randn(shape, device=device)
    time = 1.0
    traj = [x_t]
    # TODO: while time >= -dt / 2:  v_t = model(x_t, 全为 time 的 (B,) 张量, obs)；x_t = x_t + dt * v_t；time += dt
    raise NotImplementedError
    return (x_t, traj) if return_traj else x_t

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def euler_sample(model, obs, shape, num_steps=10, return_traj=False):
    dt = -1.0 / num_steps
    x_t = torch.randn(shape, device=device)
    time = 1.0
    traj = [x_t]
    while time >= -dt / 2:
        v_t = model(x_t, torch.full((shape[0],), time, device=device), obs)
        x_t = x_t + dt * v_t
        time += dt
        traj.append(x_t)
    return (x_t, traj) if return_traj else x_t

In [ ]:
# ✅ 自测：步数的影响
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, n_steps in zip(axes, [1, 2, 5, 10]):
    s = euler_sample(model, dummy(3000), (3000, 1, 2), n_steps)[:, 0].cpu()
    on_ring = ((s.norm(dim=-1) - 1).abs() < 0.25).float().mean().item()
    ax.scatter(*s.T, s=1, c="C3"); ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal")
    ax.set_title(f"{n_steps} Euler steps: on ring {on_ring:.2f}")
plt.tight_layout(); plt.show()
s10 = euler_sample(model, dummy(3000), (3000, 1, 2), 10)[:, 0]
assert ((s10.norm(dim=-1) - 1).abs() < 0.25).float().mean() > 0.85
s1 = euler_sample(model, dummy(3000), (3000, 1, 2), 1)[:, 0]
print("1 步采样的平均半径:", s1.norm(dim=-1).mean().item().__round__(3), "（所有点挤在圆心附近）")
print("通过 ✅")

**为什么 1 步会失败？** $t=1$ 时 $\mathbf x_1=\boldsymbol\epsilon$ 与数据无关，所以最优速度是 $\mathbf v^*(\boldsymbol\epsilon,1)=\boldsymbol\epsilon-\mathbb E[\mathbf x]$。走一整步 $\Delta t=-1$ 得到 $\boldsymbol\epsilon-\mathbf v^*=\mathbb E[\mathbf x]$，**正好是数据的均值**。这和 MSE 回归的"取平均"是同一个问题！多走几步，每一步只走一小段，后面的步骤就能根据已经"偏向"的方向修正，最终落到某个模式上。

### 看看流动的轨迹

In [ ]:
_, traj = euler_sample(model, dummy(300), (300, 1, 2), 10, return_traj=True)
traj = torch.stack(traj)[:, :, 0].cpu()                    # (11, 300, 2)
plt.figure(figsize=(4.5, 4.5))
for i in range(300):
    plt.plot(traj[:, i, 0], traj[:, i, 1], "-", c="gray", lw=0.5, alpha=0.5)
plt.scatter(*traj[0].T, s=3, c="blue", label="t=1 (noise)"); plt.scatter(*traj[-1].T, s=3, c="red", label="t=0 (data)")
plt.legend(); plt.gca().set_aspect("equal"); plt.title("flow trajectories (10 Euler steps)"); plt.show()

轨迹接近直线：这正是用"直线插值"定义概率路径的好处，也是少量步数就够用的原因。

---

## 3. 流匹配策略：在双峰障碍物任务上

把上面的 `fm_loss` 和 `euler_sample` 用在动作块上，就是一个**流匹配策略**：π0 动作头的最小版本。

In [ ]:
from vla_ladder.envs import ObstacleReachEnv, collect_demos
import time as _time

H, EXEC = 16, 8
env = ObstacleReachEnv(num_envs=100, seed=0)
demos = collect_demos(env, n_episodes=200, seed=0)
Os, As = [], []
for d in demos:
    T_ep = len(d["actions"]); idx = np.minimum(np.arange(T_ep)[:, None] + np.arange(H), T_ep - 1)
    Os.append(d["obs"]); As.append(d["actions"][idx])
O = torch.tensor(np.concatenate(Os)).to(device); A = torch.tensor(np.concatenate(As)).to(device)

set_seed(0)
policy = ChunkDenoiser(obs_dim=2, act_dim=2, horizon=H).to(device)
opt = torch.optim.AdamW(policy.parameters(), 1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 4000)
for step in range(4000):
    i = torch.randint(0, len(O), (256,), device=device)
    loss = fm_loss(policy, A[i], O[i])
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
policy.eval()

@torch.no_grad()
def evaluate(num_steps, n=200, seed=7):
    env = ObstacleReachEnv(num_envs=n, seed=seed); obs = env.reset(); trajs = [obs.copy()]
    t0 = _time.time()
    while not env.done.all():
        chunk = euler_sample(policy, torch.tensor(obs, device=device), (n, H, 2), num_steps).clamp(-1, 1).cpu().numpy()
        for k in range(EXEC):
            obs, _, _, info = env.step(chunk[:, k]); trajs.append(obs.copy())
    traj = np.stack(trajs, 1); succ = info["success"]
    left = (traj[:, :, 0].min(1) < -0.3)[succ].mean() if succ.any() else float("nan")
    return {"success": succ.mean(), "left": left, "time": _time.time() - t0}, traj

for n_steps in [1, 3, 10]:
    r, traj = evaluate(n_steps)
    print(f"Euler {n_steps:2d} 步：成功率 {r['success']:.2f}，左绕比例 {r['left']:.2f}，评估耗时 {r['time']:.2f}s")
r, traj = evaluate(10)
assert r["success"] > 0.9
env.plot(list(traj[:60]), title=f"flow matching policy (10 steps): success={r['success']:.2f}"); plt.show()

1 步时成功率接近 0（走"平均动作"撞上障碍物），10 步时接近 100%，并且两种模式都保留。对比上一课 DDPM 的 100 步，**流匹配用 1/10 的计算量得到了同样的效果**。

---

## 4. π0 的流匹配和本课有什么不同？

| | 本课 | π0（openpi） |
|---|---|---|
| 公式 | $\mathbf x_t=t\boldsymbol\epsilon+(1-t)\mathbf x$，$\mathbf u=\boldsymbol\epsilon-\mathbf x$ | 完全相同 |
| 时间采样 | Beta(1.5,1)·0.999+0.001 | 完全相同 |
| 推理 | 10 步 Euler，$t$: 1→0 | 完全相同 |
| 时间编码 | 正弦编码 + MLP | 正弦编码（周期 4e-3~4.0）；π0：与动作拼接后过 MLP；**π0.5：通过 adaRMSNorm 注入每一层** |
| 速度网络 | 残差 MLP，观测通过 FiLM 注入 | **动作专家 Transformer**（Gemma-300M），动作 token 通过**注意力**读取图像+文字+状态 |
| 动作块 | 16 步 × 2 维 | 50 步 × 32 维（真实维度不够 32 的补零） |
| 观测编码只算一次 | 每步都重新编码观测（MLP 很便宜） | 前缀（图像+文字）KV cache 只算一次，10 步去噪复用 |

下一章，我们把本课的 MLP 速度网络换成"与 VLM 共享注意力的动作专家 Transformer"，就得到 mini-π0。

## 小结

- 流匹配：直线路径 $\mathbf x_t=t\boldsymbol\epsilon+(1-t)\mathbf x$，回归速度 $\boldsymbol\epsilon-\mathbf x$，推理时从 $t=1$ Euler 积分到 $t=0$。
- 回归到条件速度的期望，就能得到正确的生成模型；loss 不会降到 0。
- 1 步积分 = 取平均（失败）；10 步足够（π0 默认值）。
- π0 的 Beta(1.5,1) 时间采样把训练重心放在高噪声区域。